# Driver Drowsiness — Google Colab

Two things, **top to bottom**:

1. **Run the detector on a clip you upload** — the same pipeline as the live
   system (`main.py`), audio **and** video, returning an **annotated video** with
   the on-screen HUD.
2. *(Optional)* **Calibrate the thresholds to your own labelled data.**

**The project code is pulled straight from GitHub** — no Google Drive for the main
flow. You set your repo URL in the next cell; the clip you analyse is uploaded from
your computer. Only the *optional* calibration in section 3 needs a labelled video
dataset, which (being large) still lives in Drive.

> **Prerequisite:** push this project to a GitHub repo first — including the new
> `alertVSdrowsy/file_audio_model.py` and the updated `main.py` / `audio_model.py`.


## 1 · Setup (run once)

In [ ]:
# Get the project from GitHub (no Google Drive needed for the main flow).
REPO_URL = 'https://github.com/<you>/<repo>.git'   # <-- EDIT THIS to your repo
# Private repo? Use a personal-access-token URL instead:
#   REPO_URL = 'https://<GITHUB_TOKEN>@github.com/<you>/<repo>.git'

import os, glob
get_ipython().system('rm -rf /content/repo')
get_ipython().system(f'git clone --depth 1 "{REPO_URL}" /content/repo')

# Auto-find the alertVSdrowsy folder inside the clone (the dir that holds main.py +
# config.py + file_audio_model.py) - works whether the repo IS the project or
# contains it in a subfolder.
cand = [os.path.dirname(p)
        for p in glob.glob('/content/repo/**/main.py', recursive=True)
        if {'config.py', 'file_audio_model.py'} <= set(os.listdir(os.path.dirname(p)))]
assert cand, 'Could not find alertVSdrowsy (main.py + config.py + file_audio_model.py) in the repo.'
PROJECT = cand[0]
print('project ->', PROJECT)

In [ ]:
# Install the runtime deps (the clone above already provides the code).
# - mediapipe/opencv/numpy: the video sense (no TensorFlow; _tf_guard handles that).
# - panns-inference: the SAME audio tagger the live mic uses, run on the clip's
#   soundtrack so a recorded run matches the live audio+video verdict.
# Tip: for speed pick a GPU runtime (Runtime > Change runtime type > T4 GPU).
!pip -q install mediapipe opencv-python numpy panns-inference

## 2 · Run the detector on an uploaded clip

Runs `main.py` headless on your clip **exactly like the live system** — the same
video sense **and** the same audio sense — and burns the live HUD (state,
confidence, EAR/MAR/MMS, red alert bar) into an output MP4. `config.py` is imported
automatically and applies `learned_thresholds.json` if you create one in section 3.

**Audio + video parity:** there's no microphone on Colab, so `--audio-file` feeds
the clip's **own soundtrack** through the same PANNs tagger the mic uses. That
restores the tie-break the camera can't make alone — an open mouth that *sounds*
like talking → ALERT, one that's silent/yawning → DROWSY.

> First run downloads the PANNs model (~300 MB) — a few minutes, once per session.
> A GPU runtime makes tagging much faster. If the clip has no audio track, the run
> falls back to video-only automatically (it prints the reason).

In [ ]:
# Upload ONE video clip (mp4/avi/mov/mkv...).
from google.colab import files
up = files.upload()                      # pick a single video file
VIDEO = '/content/' + next(iter(up))     # its path in Colab
print('uploaded ->', VIDEO)

# --- OR skip the upload and use a clip already in Drive: ---
# VIDEO = '/content/drive/MyDrive/dms_data/Yawning/clip1.mp4'

In [ ]:
# Run the detector EXACTLY like live, headless, on the clip.
# Flags: --source (video, not webcam), --audio-file (the clip's own soundtrack,
# not a mic), --no-display (no window), --save-video (burn the HUD into an MP4).
OUT = '/content/annotated.mp4'

# Start from a clean log so the summary below reflects only THIS clip.
get_ipython().system(f'rm -f "{PROJECT}/logs/events.csv" "{PROJECT}/logs/events.jsonl"')

get_ipython().system(
    f'cd "{PROJECT}" && python main.py --source "{VIDEO}" '
    f'--audio-file "{VIDEO}" --no-display --save-video "{OUT}"')
print('annotated video ->', OUT)

In [ ]:
# Re-encode to H.264 (cv2 writes 'mp4v', which the browser player often won't
# show), then play it inline with the HUD burned in - the same view you'd see live.
H264 = '/content/annotated_h264.mp4'
get_ipython().system(f'ffmpeg -y -loglevel error -i "{OUT}" -vcodec libx264 -pix_fmt yuv420p "{H264}"')

from IPython.display import HTML
from base64 import b64encode
mp4 = open(H264, 'rb').read()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,'
     f'{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
# The event log the detector wrote (the SAME file the live system writes).
# DROWSY rows are the moments it flagged the driver; 'reasons' says why.
import pandas as pd
ev = pd.read_csv(f'{PROJECT}/logs/events.csv')
print(f'{len(ev)} events logged.  DROWSY events:')
display(ev[ev.state == 'DROWSY'][['timestamp', 'confidence', 'reasons', 'ear', 'mar']]
        .reset_index(drop=True))

# Download the annotated video to your machine.
from google.colab import files
files.download(H264)

---
## 3 · (Optional) Calibrate thresholds to your own data

Everything below is **optional**. It fits `MAR_YAWN`, `MMS_YAWN_LEVEL` and
`EAR_CLOSED` to YOUR labelled clips and writes `learned_thresholds.json` next to
`config.py`; section 2 then picks it up automatically on its next run.

Needs a labelled video dataset `dms_data/<Label>/*.mp4`. Video datasets are large,
so this section mounts **Google Drive** for the *data only* — the code still comes
from the GitHub clone in section 1. It reuses the **same feature maths** as the live
detector, so a threshold you fit means exactly the same thing at runtime.

In [ ]:
# Mount Drive for the labelled video DATASET (the code already came from GitHub).
from google.colab import drive
drive.mount('/content/drive')

# Extract per-frame features from ALL labelled clips (CPU-heavy - the slow part).
# features.csv is GENERATED here; it is not a file that ships with the repo.
DATA_DIR = '/content/drive/MyDrive/dms_data'   # your labelled clips (in Drive)
OUT_CSV  = '/content/features.csv'             # created by this cell

get_ipython().system(f'cd "{PROJECT}/train" && python extract_features.py --videos "{DATA_DIR}" --out "{OUT_CSV}"')
print('features ->', OUT_CSV)

In [ ]:
# Load the features and look at each class. Only frames with a detected face
# matter for fitting thresholds.
# NOTE: if you ran annotate_yawns.py, a 'Yawning' clip contributes 'Yawning' rows
# only for the marked yawn span; its closed-mouth frames appear as 'Neutral' and
# IGNORE frames were dropped. So expect a 'Neutral' class and a cleaner 'Yawning'.
import pandas as pd
df = pd.read_csv(OUT_CSV)
df = df[df.face_found == 1].copy()
print('frames per class:')
print(df.label.value_counts())
df.groupby('label')[['ear', 'mar', 'lip_gap_level']].describe().T

In [ ]:
# A tiny helper: pick the threshold that best separates two groups.
# 'greater' means the positive class has LARGER values (e.g. MAR for yawning);
# 'less' means SMALLER (e.g. EAR for closed eyes). We scan candidate cut points
# and keep the one with the best balanced accuracy (Youden's J).
import numpy as np

def best_threshold(pos, neg, direction='greater'):
    pos, neg = np.asarray(pos), np.asarray(neg)
    cands = np.unique(np.concatenate([pos, neg]))
    best_t, best_j = None, -1
    for t in cands:
        if direction == 'greater':
            tpr = (pos >= t).mean(); fpr = (neg >= t).mean()
        else:
            tpr = (pos <= t).mean(); fpr = (neg <= t).mean()
        j = tpr - fpr
        if j > best_j:
            best_j, best_t = j, float(t)
    return best_t, best_j

In [ ]:
# Fit the three core thresholds from the data.
def vals(labels, col):
    return df[df.label.isin(labels)][col].values

# 'Neutral' (from annotate_yawns.py) = closed-mouth frames OUTSIDE the marked yawn
# span. They are training NEGATIVES, so they join the negative pools below.
# (No 'Neutral' rows if you haven't annotated -> harmless no-op.)

# MAR_YAWN: yawning mouth is wide open vs everything else.
mar_t, mar_j = best_threshold(vals(['Yawning'], 'mar'),
                              vals(['Singing', 'Alert', 'Distracted', 'Neutral'], 'mar'),
                              'greater')

# MMS_YAWN_LEVEL: yawning lip-gap level vs talking/singing/closed-mouth.
lvl_t, lvl_j = best_threshold(vals(['Yawning'], 'lip_gap_level'),
                              vals(['Singing', 'Alert', 'Neutral'], 'lip_gap_level'),
                              'greater')

# EAR_CLOSED: eyes-closed states (Sleeping/Drowsy) vs eyes-open (Alert).
ear_t, ear_j = best_threshold(vals(['Sleeping', 'Drowsy'], 'ear'),
                              vals(['Alert'], 'ear'),
                              'less')

print(f'MAR_YAWN       = {mar_t:.3f}  (separation J={mar_j:.2f})')
print(f'MMS_YAWN_LEVEL = {lvl_t:.3f}  (separation J={lvl_j:.2f})')
print(f'EAR_CLOSED     = {ear_t:.3f}  (separation J={ear_j:.2f})')

In [ ]:
# Save the fitted thresholds next to config.py. The detector loads this file at
# startup and overrides its defaults - no code change needed. Only keep a key if
# its separation J is decent, so a noisy fit can't make things worse than defaults.
import json, os
learned = {}
if mar_j > 0.5: learned['MAR_YAWN'] = round(mar_t, 3)
if lvl_j > 0.5: learned['MMS_YAWN_LEVEL'] = round(lvl_t, 3)
if ear_j > 0.5: learned['EAR_CLOSED'] = round(ear_t, 3)

dest = os.path.join(PROJECT, 'learned_thresholds.json')
with open(dest, 'w') as f:
    json.dump(learned, f, indent=2)
print('wrote', dest)
print(json.dumps(learned, indent=2))
print('\nRe-run section 2 in THIS session to apply these automatically.')
print('The clone is temporary, so to keep them, commit learned_thresholds.json')
print('next to config.py in your GitHub repo (or download it):')
from google.colab import files
files.download(dest)